# 05 — Bootstrap few-shot: the program that teaches itself

**What you'll learn**

- The bootstrapping trick behind the original DSPy paper: run your program as its own *teacher*, keep the executions your metric blesses, attach them as per-predictor demos.
- The `trace` contract — the metric's second job as a quality gate.
- How to *read what was learned* (`named_predictors`, `save()`), and round-trip it with `load()`.
- Turning one attempt into a search: `BootstrapFewShotWithRandomSearch`.
- The paper ↔ code map for *DSPy: Compiling Declarative Language Model Calls into Self-Improving Pipelines* (Khattab et al., 2023).

*Time: ~30 min. Cost: ~$0.15–0.30 (this is the first notebook that runs an optimizer). Cached reruns are free.*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

# Optional tracing for any notebook (taught in notebook 11) — pick ONE option:
# Option A, MLflow: logs land in ./mlflow.db; notebook 11 launches the UI from a cell.
#   import mlflow
#   mlflow.set_experiment("dspy-lab")
#   mlflow.dspy.autolog()
# Option B, Arize Phoenix: needs the Phoenix server up (see notebook 11's launcher cell).
#   from phoenix.otel import register
#   register(project_name="dspy-lab", auto_instrument=True)

Boilerplate — same code as notebooks 03 (world + program) and 04 (data + metric). Skim and run.

In [ ]:
# === boilerplate: aldervane micro-world — same code as notebook 03 ===
import json, math, re
from collections import Counter
from pathlib import Path

CORPUS = json.loads(Path("data/corpus.json").read_text())

STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "did", "do", "does", "each", "every",
    "for", "from", "he", "her", "his", "in", "is", "it", "its", "of", "on",
    "or", "she", "that", "the", "this", "to", "was", "were", "what", "when",
    "where", "which", "who", "whom", "whose", "with",
}

def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    # naive stemming: "captains" -> "captain", "records" -> "record"
    words = [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w
             for w in words]
    return [w for w in words if w not in STOPWORDS]

DOC_TOKENS = {p["id"]: set(tokenize(p["title"] + " " + p["text"])) for p in CORPUS}
IDF = Counter()
for tokens in DOC_TOKENS.values():
    IDF.update(tokens)
IDF = {t: math.log(len(CORPUS) / df) for t, df in IDF.items()}

def search(query, k=2):
    """Return the k corpus passages that best match the query (keyword + IDF)."""
    q = set(tokenize(query))
    ranked = sorted(
        CORPUS,
        key=lambda p: (-sum(IDF.get(t, 0.0) for t in q & DOC_TOKENS[p["id"]]), p["id"]),
    )
    return ranked[:k]

def as_context(passages):
    return [f"{p['title']}: {p['text']}" for p in passages]

class HopQA(dspy.Module):
    """Two-hop retrieval QA over the Aldervane corpus."""

    def __init__(self, k=2):
        super().__init__()
        self.k = k
        self.hop1 = dspy.Predict("question -> search_query")
        self.hop2 = dspy.Predict("question, context: list[str] -> search_query")
        self.answer = dspy.ChainOfThought("question, context: list[str] -> answer")

    def forward(self, question):
        q1 = self.hop1(question=question).search_query
        first = search(q1, k=self.k)
        q2 = self.hop2(question=question, context=as_context(first)).search_query
        second = search(q2, k=self.k)
        passages = {p["id"]: p for p in first + second}      # merge + dedupe
        pred = self.answer(question=question, context=as_context(list(passages.values())))
        return dspy.Prediction(
            answer=pred.answer,
            reasoning=pred.reasoning,
            queries=[q1, q2],
            retrieved_ids=sorted(passages),
        )

In [ ]:
# === boilerplate: data + metric — same code as notebook 04 ===
QA = json.loads(Path("data/qa.json").read_text())

def to_examples(rows):
    return [dspy.Example(**row).with_inputs("question") for row in rows]

train, dev, test = (to_examples(QA[s]) for s in ("train", "dev", "test"))

def norm(s):
    s = re.sub(r"[^a-z0-9\s]", " ", s.lower())
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def em_metric(example, pred, trace=None):
    """1.0 if the predicted answer exactly matches the gold answer (after normalization)."""
    return norm(pred.answer) == norm(example.answer)

Re-establish the baseline. Thanks to the disk cache, re-scoring the same program on the same questions costs nothing:

In [ ]:
evaluate = dspy.Evaluate(devset=dev, metric=em_metric, num_threads=8, display_progress=True)

baseline_score = evaluate(HopQA()).score
baseline_score

## The idea

Recall the two facts we planted earlier: a predictor's demos are just `Example`s (notebook 02), and a `Prediction` *is* an `Example` (notebook 02, closing trick). So outputs can become demos. Bootstrapping is the algorithm hiding in that pun:

```
                      +--------------------------------------+
   trainset --------> |  teacher = a copy of YOUR program    |
   (24 questions)     |  runs each question, trace recorded  |
                      +------------------+-------------------+
                                         |  full trace: every predictor's
                                         |  (inputs -> outputs), per question
                                         v
                      +--------------------------------------+
                      |  metric(example, pred, trace) ?      |   <- the quality gate
                      +---------+---------------------+------+
                            passed                 failed
                               |                      |
                               v                      v
                  each predictor's step from      discarded
                  the trace becomes one of
                  its demos (up to a cap)
                               |
                               v
                      student = your program, demos attached per predictor
```

Read the flow twice, because the twist is easy to miss: for a 2-hop question the teacher answered correctly, the trace *contains a working hop-2 query* — one that searched for the bridge entity. That intermediate step, which **no dataset ever labeled**, becomes a worked example attached to `hop2`. The program manufactures supervision for its own internal steps.

If you want the ML analogy: this is self-distillation / pseudo-labeling, with your metric as the filter. And note what it is *not* — there is no gradient anywhere. We are doing discrete search over prompt space, scored by the metric.

## The metric's second job: the `trace` contract

Notebook 04 promised an explanation of `metric(example, pred, trace=None)`. Here it is: **the bootstrapper is who passes `trace`.** During compilation, when the teacher finishes a question, your metric is called with the full execution trace, and its verdict decides whether that execution enters the demo pool. Different question, different contract:

| caller | `trace` | question being asked | you return |
|---|---|---|---|
| `dspy.Evaluate` | `None` | "how good is this prediction?" | a score (float/bool) |
| bootstrapping | the execution record | "is this execution clean enough to *imitate*?" | a strict `bool` |

The canonical pattern makes both jobs explicit:

In [ ]:
def qa_metric(example, pred, trace=None):
    """Normalized exact match; strict gate when bootstrapping."""
    correct = norm(pred.answer) == norm(example.answer)
    if trace is not None:        # bootstrapping: admit only flawless executions
        return correct
    return float(correct)        # evaluation: a score

For exact match the two branches happen to agree — but keep the pattern. The moment your evaluation score has shades of gray (partial credit, semantic similarity), the `trace` branch is where you stay ruthless, because **every demo you admit is a behavior you clone.**

## Compile

The optimizer takes the metric and two caps: how many bootstrapped (self-generated) demos, and how many labeled (plain trainset) demos to attach per predictor.

In [ ]:
tp = dspy.BootstrapFewShot(
    metric=qa_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=4,
)

boot = tp.compile(HopQA(), trainset=train)

> **What you should see:** a progress bar while the teacher works through training questions; it stops early once enough passing traces are collected. Your original program is untouched — `compile` returned a **new** one (`boot`) with filled demo slots. That is the universal optimizer shape from notebook 02: `optimized = optimizer.compile(program, trainset=...)`.

In [ ]:
boot_score = evaluate(boot).score
print(f"baseline {baseline_score}  ->  bootstrapped {boot_score}")

> **What you should see:** an improvement, typically **+10 to +25 points** on this task (recall: 16 dev questions, 6.25 points each — expect noise).
>
> | program | dev EM (16 q) | expect | notebook |
> |---|---|---|---|
> | `HopQA()` zero-shot | *yours* | ~25–55 | 04 |
> | + `BootstrapFewShot` | *yours* | baseline +10–25 | 05 |
>
> **If it didn't improve:** (a) score stuck at 0 → your model is answering verbosely and `norm` can't save it; check three failures by hand. (b) score unchanged and the next cell shows **0 demos** attached → the metric rejected every teacher trace; the teacher needs help — see the `teacher_settings` exercise. (c) small dip → toy-scale variance; run the random-search variant below before concluding anything.

## Read what was learned

Never trust an optimizer report; read the parameters. First, the shape of the update, then an actual learned demo:

In [ ]:
for name, predictor in boot.named_predictors():
    print(f"{name:15s} {len(predictor.demos)} demos")

print("\ndemos attached to hop2:")
for demo in boot.hop2.demos:
    learned_query = getattr(demo, "search_query", None)     # only bootstrapped demos have one
    if learned_query is not None:
        print(f"  [bootstrapped] Q: {demo.question}")
        print(f"                 learned search_query: {learned_query!r}")
    else:
        print(f"  [labeled]      Q: {demo.question}")

> **What you should see:** each predictor now carries demos, and — the payoff of the whole notebook — at least one `hop2` bootstrapped demo whose `search_query` is **not a restated question but a bridge entity** (a name like `"Mara Venn"` or `"Kelvane Foundry"`). Nobody wrote that query. The teacher stumbled into the behavior, the metric certified the outcome, and the demo now teaches it on every future call.
>
> This is the fix for the exact failure you diagnosed in notebook 04's error analysis — delivered as *data*, not as a prompt edit.

In [ ]:
import json
from pathlib import Path

Path("artifacts").mkdir(exist_ok=True)
boot.save("artifacts/05_bootstrap.json")

state = json.loads(Path("artifacts/05_bootstrap.json").read_text())
print(list(state.keys()))

> **What you should see:** the `state_dict()` ritual from notebook 02, now with real content — per-predictor entries holding `demos` and `signature` (instructions). Open the file in an editor and read `hop2`'s demos; the learned query from the previous cell is sitting there as plain JSON.
>
> Sidebar — two save formats: `save("x.json")` stores **state only** (readable, portable; load it into a freshly constructed program). `save("dir/", save_program=True)` pickles the **whole program** via cloudpickle; loading that requires `dspy.load("dir/", allow_pickle=True)` — 3.x makes you opt into pickle explicitly, since unpickling runs code. For learning (and most deployment), the JSON is the one you want.

In [ ]:
fresh = HopQA()
fresh.load("artifacts/05_bootstrap.json")
evaluate(fresh).score          # cache makes this instant — expect the identical score

## From one attempt to a search

`BootstrapFewShot` made *one* program: it took passing traces in the order found. But which 4 demos land on which predictor is a choice, and some subsets teach better than others. `BootstrapFewShotWithRandomSearch` (alias `dspy.BootstrapRS`) makes N candidate programs — different seeds, different demo subsets — evaluates each, and keeps the winner. This is the moment "optimization" stops being a metaphor: candidates × evaluations, guided by the metric.

*Cost note: ~`num_candidate_programs` × (bootstrap + a dev evaluation) — with the settings below, a few hundred LM calls, mostly cache-cold. Expect a few minutes.*

In [ ]:
rs = dspy.BootstrapFewShotWithRandomSearch(
    metric=qa_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=4,
    num_candidate_programs=4,
    num_threads=8,
)

boot_rs = rs.compile(HopQA(), trainset=train, valset=dev)
rs_score = evaluate(boot_rs).score
print(f"baseline {baseline_score} -> bootstrap {boot_score} -> +random search {rs_score}")

> **What you should see:** per-candidate scores scrolling by, then the best kept. Typically a few points above plain bootstrap — sometimes a tie at this toy scale; the *shape* (generate candidates → score → select) is the lesson, and it is the skeleton every later optimizer elaborates.
>
> Add your number to the leaderboard.

## Paper ↔ code: the DSPy compiler paper

This notebook implements the core of *DSPy: Compiling Declarative Language Model Calls into Self-Improving Pipelines* (Khattab et al., 2023, [arXiv:2310.03714](https://arxiv.org/abs/2310.03714)). The map:

| Paper concept | Plain English | Where it lives in code |
|---|---|---|
| Signature | Declarative typed I/O contract for an LM call | `dspy.Signature`, `"question -> search_query"` (notebook 01) |
| Parameterized declarative module | A pipeline whose prompts are learnable state, not string literals | `dspy.Module`; per-predictor `demos` + `signature.instructions` (θ, notebook 02) |
| **Teleprompter** | The optimizer that compiles a program against a metric | the classes in `dspy.teleprompt` — the paper's coinage survives in the import path |
| Compiling | Search over prompt-space parameter values; returns an improved program | `optimizer.compile(program, trainset=...)` |
| Bootstrapped demonstrations (the *demonstrate* stage) | Teacher (a copy of your program) runs inputs; metric-passing traces become per-predictor few-shot demos | `dspy.BootstrapFewShot(metric=..., max_bootstrapped_demos=..., teacher_settings=...)`; the metric's `trace is not None` branch |
| Labeled demonstrations | Plain training pairs pasted in as demos | `dspy.LabeledFewShot(k)` (notebook 02); `max_labeled_demos` here |
| Random search over candidate pipelines | Bootstrap many programs, keep the best on validation | `dspy.BootstrapFewShotWithRandomSearch(num_candidate_programs=...)` |
| Validation logic / metric | The accept/score function steering compilation | `qa_metric(example, pred, trace=None)` |

One sentence of lineage: the paper's thesis was that *pipelines whose prompts are parameters can compile themselves from a handful of examples* — this notebook is that thesis, run end to end on a world the model had never seen.

## Recap

| Concept | One-liner |
|---|---|
| bootstrapping | teacher-run traces, filtered by the metric, become per-predictor demos |
| the twist | intermediate steps (hop-2 queries) get supervision no dataset contained |
| `trace` contract | `None` → score it; set → strict bool gate on what gets imitated |
| `compile` | returns a new program; original untouched |
| read your θ | `named_predictors()` + `save("x.json")` — always inspect the learned demos |
| `BootstrapFewShotWithRandomSearch` | N candidates → evaluate → keep the best (`dspy.BootstrapRS`) |

## Exercises

1. Recompile with `max_bootstrapped_demos=8`. Compare the dev score *and* the prompt length (`inspect_history`) — demos are tokens; where does the trade-off stop paying?
2. Give the teacher a stronger brain: `dspy.BootstrapFewShot(metric=qa_metric, teacher_settings=dict(lm=dspy.LM(STRONG_MODEL)), ...)`. The strong model writes the traces; your cheap model inherits them as demos. (This is distillation — compare scores.)
3. Ablation: `boot.hop2.demos = []`, re-evaluate. How much of the total gain lived in that one predictor? You just did per-module credit assignment by hand.

**Next up:** demos teach by example. Notebook 06 teaches by *description* — optimizers that rewrite the instructions themselves (COPRO, and the Bayesian search of MIPROv2).